In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)

df_closed = df[
    df["ISSUE"].str.contains("closed", case=False, na=False)
]


df_closed =  df_closed[['ISSUE', 'Network Name','Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)


df_closed["Station ID"] = pd.to_numeric(df_closed["Station ID"], errors="coerce").astype("Int64")

len(df_closed)

df_closed

,ISSUE,Network Name,Station ID,Unique Names,Native ID
0,Site CLOSED - Move all data collected post Oct...,BCH-GSO-HMP -> ENV ASP,2436,Brenda Mines,BMN


In [5]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
    FROM meta_history m_old
    JOIN obs_raw o_old ON m_old.history_id = o_old.history_id
    JOIN meta_history m_new ON TRUE
    JOIN obs_raw o_new ON m_new.history_id = o_new.history_id
    AND o_old.obs_time = o_new.obs_time
    AND o_old.vars_id  = o_new.vars_id
    WHERE m_old.station_id = %s
    AND m_new.station_id = %s
    AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def get_station_overlap(old_station_id, new_station_id, engine):
    params = (old_station_id, new_station_id,
              old_station_id, new_station_id,
              old_station_id, new_station_id)
    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = get_station_overlap(2589, 12579, engine)
print(stats)

n_old                   456192
n_new                     5662
n_overlap                    0
n_overlap_same_datum         0
Name: 0, dtype: int64


In [6]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
    FROM meta_history m_old
    JOIN obs_raw o_old ON m_old.history_id = o_old.history_id
    JOIN meta_history m_new ON TRUE
    JOIN obs_raw o_new ON m_new.history_id = o_new.history_id
       AND o_old.obs_time = o_new.obs_time
       AND o_old.vars_id  = o_new.vars_id
    WHERE m_old.station_id = %s
      AND m_new.station_id = %s
      AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum,

    -- oldest obs_time for old station
    (SELECT MIN(o.obs_time)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS oldest_old_obs_time,

    -- newest obs_time for old station
    (SELECT MAX(o.obs_time)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS newest_old_obs_time,

    -- oldest obs_time for new station
    (SELECT MIN(o.obs_time)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS oldest_new_obs_time,

    -- newest obs_time for new station
    (SELECT MAX(o.obs_time)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS newest_new_obs_time
"""

def get_station_overlap(old_station_id, new_station_id, engine):
    params = (
        old_station_id,  # n_old
        new_station_id,  # n_new
        old_station_id,  # n_overlap (old)
        new_station_id,  # n_overlap (new)
        old_station_id,  # n_overlap_same_datum (old)
        new_station_id,  # n_overlap_same_datum (new)
        old_station_id,  # oldest_old_obs_time
        old_station_id,  # newest_old_obs_time
        new_station_id,  # oldest_new_obs_time
        new_station_id   # newest_new_obs_time
    )
    
    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

old_id = 2589
new_id = 12579

stats = get_station_overlap(old_id, new_id, engine)

print(f"Old station ID: {old_id}")
print(f"New station ID: {new_id}")

print("Old station count:", stats.n_old)
print("New station count:", stats.n_new)
print("Overlap (time+variable):", stats.n_overlap)
print("Overlap identical datum:", stats.n_overlap_same_datum)

print("Old station oldest obs_time:", stats.oldest_old_obs_time)
print("Old station newest obs_time:", stats.newest_old_obs_time)

print("New station oldest obs_time:", stats.oldest_new_obs_time)
print("New station newest obs_time:", stats.newest_new_obs_time)


Old station ID: 2589
New station ID: 12579
Old station count: 456192
New station count: 5662
Overlap (time+variable): 0
Overlap identical datum: 0
Old station oldest obs_time: 1985-10-17 16:00:00
Old station newest obs_time: 2009-06-30 23:00:00
New station oldest obs_time: 2020-07-27 00:00:00
New station newest obs_time: 2020-11-21 23:00:00


### Delete the new station

In [7]:
import sqlalchemy as sa

station_id = '14958'   # ← the ONE station you want to delete
# station_id = '12579'   # ← the ONE station you want to delete

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    res_flags = conn.execute(delete_obs_flags, {"station_id": station_id})
    res_obs   = conn.execute(delete_obs, {"station_id": station_id})
    res_dv    = conn.execute(delete_obs_derived, {"station_id": station_id})
    res_hist  = conn.execute(delete_history, {"station_id": station_id})
    res_sta   = conn.execute(delete_station, {"station_id": station_id})

print(
    f"Station {station_id} deleted: "
    f"flags={res_flags.rowcount}, "
    f"obs_raw={res_obs.rowcount}, "
    f"obs_derived={res_dv.rowcount}, "
    f"meta_history={res_hist.rowcount}, "
    f"meta_station={res_sta.rowcount}"
)

Station 14958 deleted: flags=0, obs_raw=182137, obs_derived=0, meta_history=1, meta_station=1


### The last closed

In [ ]:
Additional - Responsibility Transfer - Move any data from 35092 collected post October 1, 2022 to this record. 5	-> BCH-GSO-HMP	-> MOY		-> Moyie Mountain	-> 115.76733 W, 49.2521667 N, Elev. 1930

CLOSED - responsibility change - Move any data from this site collected from October 1, 2022 on to New addditional station (line 1932 - "Moyie Mountain", BC H)
14	BC-Snow	2C10P 	2549	Moyie Mountain	115.767 W, 49.25 N, Elev. 1930 m

### Create the new station first

In [8]:

wanted_issues = [
    "Additional - Responsibility Transfer - Move any data from 35092 collected post October 1, 2022 to this record.",
    'CLOSED - responsibility change - Move any data from this site collected from October 1, 2022 on to New addditional station (line 1932 - "Moyie Mountain", BC H)'
]

df_3 = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_3 = df_3[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

df_3
pattern = r'->\s*([0-9\.\-]+)\s*W,\s*([0-9\.\-]+)\s*N,\s*Elev\.\s*([0-9\.\-]+)\s*'

df_3[['lon', 'lat', 'elev']] = df_3['Unique Location (String)'].str.extract(pattern).astype(float)

df_3["Native ID"] = df_3["Native ID"].str.replace(r'.*->\s*', '', regex=True)
df_3["Unique Names"] = df_3["Unique Names"].str.replace(r'.*->\s*', '', regex=True)
df_3["Network Name"] = df_3["Network Name"].str.replace(r'.*->\s*', '', regex=True)

df_3 = df_3.drop(columns=['Unique Location (String)'])

df_3

# df_3.iloc[0]

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,lon,lat,elev
0,Additional - Responsibility Transfer - Move an...,NaN,5.0,BCH-GSO-HMP,MOY,Moyie Mountain,115.76733,49.252167,1930.0


In [9]:
from sqlalchemy import func

stations_created = []

# Get the first row as a Series
row = df_3.iloc[0]

name = row['Unique Names']
nid  = row['Native ID']

# 1. Create Station
st = Station(
    native_id=nid,
    publish=True,
    network_id=14
)

session.add(st)
session.flush()  # get st.id

h = History(
    station_id=st.id,
    station_name=name,
    lat=float(row['lat']),
    lon=float(row['lon']),
    elevation=float(row['elev']),
    province="BC",
    country="CA",
    the_geom=func.ST_SetSRID(func.ST_MakePoint(float(row['lon']), float(row['lat'])), 4326)
)

session.add(h)
session.commit()
stations_created.append((name, st.id))
print("Created:", stations_created)

Created: [('Moyie Mountain', 14959)]


In [10]:
from sqlalchemy import text

# 1️⃣ Query to select old observations that need moving
SQL_TO_MOVE = text("""
SELECT o_old.history_id AS old_hist_id,
       o_old.obs_time,
       o_old.vars_id
FROM obs_raw o_old
JOIN meta_history h_old ON o_old.history_id = h_old.history_id
JOIN meta_station s_old ON h_old.station_id = s_old.station_id
WHERE s_old.native_id = :old_id

EXCEPT

SELECT o_new.history_id AS old_hist_id,
       o_new.obs_time,
       o_new.vars_id
FROM obs_raw o_new
JOIN meta_history h_new ON o_new.history_id = h_new.history_id
JOIN meta_station s_new ON h_new.station_id = s_new.station_id
WHERE s_new.native_id = :new_id
""")

# 2️⃣ Get the new station's history_id (assume we use latest)
SQL_NEW_HISTORY = text("""
SELECT history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :new_id
""")

def move_station_obs_fast(old_id, new_id, engine):
    with engine.begin() as conn:
        # get new history_id (latest)
        new_hist_id = conn.execute(SQL_NEW_HISTORY, {"new_id": new_id}).scalar()
        if new_hist_id is None:
            print(f"New station '{new_id}' has no history, skipping.")
            return 0

        # update all in one query
        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    JOIN meta_station s_old ON h_old.station_id = s_old.station_id
                    WHERE o.history_id = h_old.history_id
                      AND s_old.native_id = :old_id
                      AND o.obs_time >= '2022-10-01'  -- only move obs after this date
                      AND NOT EXISTS (
                          SELECT 1
                          FROM obs_raw o_new
                          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
                          JOIN meta_station s_new ON h_new.station_id = s_new.station_id
                          WHERE s_new.native_id = :new_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id = o.vars_id
                      )
                    RETURNING o.*
                )
                SELECT COUNT(*) FROM updated;
            """),
            {"old_id": old_id, "new_id": new_id, "new_hist_id": new_hist_id}
        ).scalar()

        print(f"Moved {n_move} rows from '{old_id}' to '{new_id}'")
        return n_move


# 4️⃣ Loop through your dataframe
results = []

old_id = df_3['Native ID'].iloc[1]
new_id = df_3['Native ID'].iloc[0]
print(f" Processing old='{old_id}' -> new='{new_id}'")
n = move_station_obs_fast(old_id, new_id, engine)
results.append(n)

# df_concat_new['n_moved'] = results
print("All done!")

IndexError: single positional indexer is out-of-bounds

In [11]:
df_3

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,lon,lat,elev
0,Additional - Responsibility Transfer - Move an...,NaN,5.0,BCH-GSO-HMP,MOY,Moyie Mountain,115.76733,49.252167,1930.0


In [13]:
from sqlalchemy import text

SQL_COUNT_OLD = text("""
SELECT COUNT(*) AS n_obs
FROM obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :old_id
""")

def count_old_station_obs(old_id, engine):
    with engine.connect() as conn:
        n_obs = conn.execute(SQL_COUNT_OLD, {"old_id": old_id}).scalar()
    return n_obs

old_id = df_3['Native ID'].iloc[1]

n_obs = count_old_station_obs(old_id, engine)
print(f"Old station {old_id} has {n_obs} observation records")

IndexError: single positional indexer is out-of-bounds

 - Move any data from 35092 collected post October 1, 2022 to this record." (-> BCH-GSO-HMP	-> MOY		-> Moyie Mountain	-> 115.76733 W, 49.2521667 N, Elev. 1930)

In [15]:
SQL_TO_MOVE = text("""
SELECT o_old.history_id AS old_hist_id,
       o_old.obs_time,
       o_old.vars_id
FROM obs_raw o_old
JOIN meta_history h_old ON o_old.history_id = h_old.history_id
WHERE h_old.station_id = :old_id

EXCEPT

SELECT o_new.history_id AS old_hist_id,
       o_new.obs_time,
       o_new.vars_id
FROM obs_raw o_new
JOIN meta_history h_new ON o_new.history_id = h_new.history_id
JOIN meta_station s_new ON h_new.station_id = s_new.station_id
WHERE s_new.native_id = :new_id
""")

SQL_NEW_HISTORY = text("""
SELECT history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :new_id
""")


def move_station_obs_fast(old_id, new_id, engine):
    with engine.begin() as conn:
        new_hist_id = conn.execute(SQL_NEW_HISTORY, {"new_id": new_id}).scalar()
        if new_hist_id is None:
            print(f"New station '{new_id}' has no history, skipping.")
            return 0

        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    WHERE o.history_id = h_old.history_id
                    AND h_old.station_id = :old_id
                    AND o.obs_time >= '2022-10-01'
                    AND NOT EXISTS (
                        SELECT 1
                        FROM obs_raw o_new
                        JOIN meta_history h_new ON o_new.history_id = h_new.history_id
                        JOIN meta_station s_new ON h_new.station_id = s_new.station_id
                        WHERE s_new.native_id = :new_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id = o.vars_id
                    )
                    RETURNING o.*
                )
                SELECT COUNT(*) FROM updated;
            """),
            {"old_id": old_id, "new_id": new_id, "new_hist_id": new_hist_id}
        ).scalar()

        print(f"Moved {n_move} rows from station {old_id} to '{new_id}'")
        return n_move

old_id = '35092'
new_id = df_3['Native ID'].iloc[0]

n = move_station_obs_fast(old_id, new_id, engine)



Moved 0 rows from station 35092 to 'MOY'
